# 02 — First End-to-End Run

Wire up the data pipeline, two agents, and the supervisor to produce a trading decision.

In [ ]:
import warnings
from pathlib import Path

import pandas as pd
import yfinance as yf

from hmats.agents import RSIAgent, SMACrossoverAgent
from hmats.coordinator import Supervisor
from hmats.data.pipeline import build_snapshot, compute_indicators

warnings.filterwarnings("ignore", category=FutureWarning)

## 1. Load data

Try to load from the parquet file saved by notebook 01. If it doesn't exist, fetch fresh data.

In [ ]:
TICKER = "BTC-USD"
parquet_path = Path.cwd().parents[2] / "data" / "raw" / "btc_daily.parquet"

if parquet_path.exists():
    print(f"Loading from {parquet_path}")
    df = pd.read_parquet(parquet_path)
else:
    print("Parquet not found — fetching fresh data")
    df = yf.download(TICKER, start="2020-01-01", interval="1d", auto_adjust=True)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df = compute_indicators(df)

print(f"Data shape: {df.shape}")
df.tail(3)

## 2. Build a MarketSnapshot

In [ ]:
snapshot = build_snapshot(TICKER, df)

print(f"Ticker:    {snapshot.ticker}")
print(f"Timestamp: {snapshot.timestamp}")
print(f"OHLCV rows: {len(snapshot.ohlcv)}")
print(f"Indicators: {list(snapshot.indicators.keys())}")

## 3. Run the Supervisor

In [ ]:
supervisor = Supervisor(agents=[SMACrossoverAgent(), RSIAgent()])
decision = supervisor.run(snapshot)

print("\n" + "=" * 60)
print(f"DECISION: {decision.action.upper()}")
print(f"Confidence: {decision.confidence:.2f}")
print(f"Position size: {decision.position_size:.2f}")
print("=" * 60)

print("\nAgent signals:")
for agent_id, signal in decision.reasoning.items():
    print(f"  {agent_id}: {signal.action} (confidence={signal.confidence:.2f})")